In [1]:
# ! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


## Data Ingestion


In [3]:
from langchain.document_loaders import TextLoader

In [6]:
loader = TextLoader("D:\\llmops_series\\data\\aigents.txt", encoding="utf8")
documents = loader.load()

In [7]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'An AI Agent is an intelligent software system capable of understanding goals, reasoning about problems, making decisions, using external tools, and executing tasks with varying degrees of autonomy.\nUnlike traditional AI applications that primarily respond to individual prompts, AI agents can carry out multistep workflows, interact with external environments, and adapt their actions based on feedback.\nFor example, a traditional chatbot might explain how to analyze a dataset. An AI agent could ret'

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [10]:
text_chunks=text_splitter.split_documents(documents)

In [11]:
text_chunks

[Document(metadata={'source': 'D:\\llmops_series\\data\\aigents.txt'}, page_content='An AI Agent is an intelligent software system capable of understanding goals, reasoning about problems, making decisions, using external tools, and executing tasks with varying degrees of autonomy.'),
 Document(metadata={'source': 'D:\\llmops_series\\data\\aigents.txt'}, page_content='Unlike traditional AI applications that primarily respond to individual prompts, AI agents can carry out multistep workflows, interact with external environments, and adapt their actions based on'),
 Document(metadata={'source': 'D:\\llmops_series\\data\\aigents.txt'}, page_content='actions based on feedback.'),
 Document(metadata={'source': 'D:\\llmops_series\\data\\aigents.txt'}, page_content='For example, a traditional chatbot might explain how to analyze a dataset. An AI agent could retrieve the dataset, validate its quality, execute Python code, generate visualizations, interpret'),
 Document(metadata={'source': 'D:\

In [ ]:
# ! uv pip install faiss-cpu


Using Python 3.12.10 environment at: /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/.venv
Resolved 3 packages in 333ms                                         
Installed 1 package in 3ms                                  
 + faiss-cpu==1.12.0


In [12]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [15]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [16]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [17]:
vectorstore

In [18]:
retriever=vectorstore.as_retriever()

In [ ]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
Key Characteristics of Agentic AI
Agentic AI systems are distinct from traditional AI models due to several core characteristics:
--------------------------------------------------
Document 2:
﻿Understanding Agentic AI
--------------------------------------------------
Document 3:
* Adaptability and Learning: They can adjust their behavior and improve their performance based on new information and experiences.
How Agentic AI Works
--------------------------------------------------
Document 4:
such as ensuring safety, interpretability, and ethical decision-making remain critical areas of research and development for the widespread adoption of agentic AI.
--------------------------------------------------


In [19]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [20]:
prompt=ChatPromptTemplate.from_template(template)

In [21]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [22]:
from langchain.schema.output_parser import StrOutputParser

In [23]:
output_parser=StrOutputParser()

In [24]:
from langchain.chat_models import ChatOpenAI

llm_model=ChatOpenAI(model_name="gpt-4.1-mini")

C:\Users\tariq\AppData\Local\Temp\ipykernel_1960\1040115773.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm_model=ChatOpenAI(model_name="gpt-4.1-mini")


In [25]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [26]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI refers to intelligent software systems, known as AI agents, that can understand goals, reason about problems, make decisions, and execute tasks autonomously. Unlike traditional AI that responds to single prompts, agentic AI can carry out multistep workflows and interact dynamically with external environments. These agents adapt their actions based on feedback and use external tools to complete complex tasks. For example, while a traditional chatbot might only explain a process, an AI agent can retrieve data, validate it, run code, generate visualizations, and interpret results. This autonomy and adaptability distinguish agentic AI from simpler AI models.'

In [27]:
import structlog